In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import yaml
from collections import defaultdict
from tqdm.notebook import tqdm
from typing import Literal

from data_utils.gecodb_compound_parser import Compound
from check_applicability import l2p_satisfaction_checkers

Load GermaNet data...: 100%|█████████▉| 99.99999999999996/100 [00:12<00:00,  7.81it/s] 
Load Wiktionary data...: 100%|██████████| 100.0/100 [00:00<00:00, 296.88it/s]            
Load Ili records...: 100%|██████████| 100.0/100 [00:00<00:00, 60857.57it/s]


In [2]:
rng = np.random.default_rng(579372518)

In [4]:
constr_path = "constraints/constraints_all_revised.yaml"
appl_ind_dir = "resources/semrel_comp_corp_study-private/constr_appl_indices"

# load constraints
with open(constr_path, encoding="utf-8") as f:
    yamls = list(yaml.safe_load_all(f))
    constraints: dict = yamls[-1]["constraints"]
    data_specs: dict = yamls[-2]["datasets"]

# load constraint applicability indices
appl_inds = {
    k: pd.read_csv(
        os.path.join(appl_ind_dir, f"{k}.tsv"),
        sep="\t",
        header=0,
        index_col=0
    )
    for k in data_specs
}

/var/folders/p7/mdc5srxs0w36vn95fwp7jq880000gn/T/ipykernel_4670/2282497725.py:12: DtypeWarning: Columns (0: p2l:decl_cl:pl:#e-0|def-0_item, 1: p2l:decl_cl:pl:#er-0/er|def-0/er_item, 2: p2l:decl_cl:pl:#er-0/er|pl_interpr-er_item, 3: p2l:decl_cl:pl:#e_uml-0|def-0_item, 4: p2l:decl_cl:pl:#0_uml-0|pl_interpr-0_uml_item, 5: p2l:decl_cl:mixed-0/s/en|pl_interpr-en_item, 6: p2l:drv:sfx:e_pl_sfx-0_item, 7: p2l:drv:deverb_schwa$-0|def-0_item, 8: p2l:drv:sfx:sfx-s|def-s_item, 9: p2l:drv:sfx:F_#in$-en_item, 10: p2l:phon_fin:sibilant$-0_item, 11: p2l:phon_fin:F_#stressed_phon$|pl_interpr-en_item, 12: p2l:phon_fin:schwa$-en|def-en_item, 13: p2l:tend:cmpx_morph-s|def-s_item, 14: l2p:en|non_par_n_pl_interpr_item, 15: l2p:e|#stressed_syl_item, 16: l2p:e|smpx_item, 17: l2p:e|!loan_item, 18: l2p:er|#stressed_syl_item, 19: l2p:er|smpx_item, 20: l2p:er|!loan_item, 21: l2p:es|isol_item, 22: l2p:ens_item, 23: l2p:pl_marker|def-pl_item) have mixed types. Specify dtype option on import or set low_memory=False.

In [ ]:
# remove unconfirmed and missing constraints
for dname, appl_ind in appl_inds.items():
    appl_ind.dropna(axis=1, how="all", inplace=True)  # drop columns with NAs
    for c_id, constr in constraints.items():
        to_remove = False
        if dname not in constr["measures"]:
            to_remove = True
            message = f"Removed {c_id} from {dname}: not implemented."
        else:
            item_reg = constr["measures"][dname]["regularity"]["item"]
            if item_reg < 0.5:
                to_remove = True
            message = f"Removed {c_id} from {dname} at item reg {item_reg}."
        if to_remove:
            appl_ind.drop(
                columns=[
                    f"{c_id}_{sfx}"
                    for sfx in ["item", "is_applicable", "applies"]
                ],
                inplace=True
            )
            print(message)

Removed p2l:decl_cl:pl:#e-0|msyl_anim-e from gecodb_v07_gn at item reg 0.088.
Removed p2l:sem:2const_anim/pers-s from gecodb_v07_gn at item reg 0.482.
Removed p2l:tend:cmpx_phon-s|def-s from gecodb_v07_gn at item reg 0.357.
Removed p2l:phon_fin:weak_F_cons$-0/s/en|!pl_interpr-0/s from sp18_pl_constr at item reg 0.416.
Removed l2p:pl_marker|def-pl from sp18_pl_constr at item reg 0.147.


In [6]:
appl_inds["gecodb_v07_gn"].head(3)

,def-0_item,def-0_is_applicable,def-0_applies,p2l:decl_cl:pl:#0-0|def-0_item,p2l:decl_cl:pl:#0-0|def-0_is_applicable,p2l:decl_cl:pl:#0-0|def-0_applies,p2l:decl_cl:pl:#s-0_item,p2l:decl_cl:pl:#s-0_is_applicable,p2l:decl_cl:pl:#s-0_applies,p2l:decl_cl:pl:#e-0|def-0_item,...,l2p:er|smpx_applies,l2p:e_uml_item,l2p:e_uml_is_applicable,l2p:e_uml_applies,l2p:es|mono_syl_item,l2p:es|mono_syl_is_applicable,l2p:es|mono_syl_applies,l2p:0_uml_item,l2p:0_uml_is_applicable,l2p:0_uml_applies
comp_gecodb,,,,,,,,,,,,,,,,,,,,,
land_+es_regierung,land_+es_regierung,True,False,land,False,NaN,land,False,NaN,land,...,NaN,land+es,False,NaN,land+es,True,True,land+es,False,NaN
land_gericht,land_gericht,True,True,land,False,NaN,land,False,NaN,land,...,NaN,land+,False,NaN,land+,False,NaN,land+,False,NaN
land_wirt,land_wirt,True,True,land,False,NaN,land,False,NaN,land,...,NaN,land+,False,NaN,land+,False,NaN,land+,False,NaN


In [7]:
class CompXConstraints:

    def __init__(self, appl_row: pd.Series, dname: str):
        # input params
        self.appl_row = appl_row
        self.dname = dname
        # comp
        self.comp_gecodb = appl_row.name
        self.comp = Compound(appl_row.name)
        self.true = self.comp.linkers[0].morph
        # acceptable (P2L-generated) linkers
        self.acceptable_linkers = defaultdict(list)
        # add the default constraint right away
        def_id = "def-0"
        def_constr = constraints[def_id]
        self.acceptable_linkers["_"].append({
            "c_id": def_id,
            "item_reg": self._reg(def_constr),
            "item_cvg": self._cvg(def_constr)
        })
        # unacceptable (L2P-restricted) linkers
        self.unacceptable_linkers = defaultdict(list)
        self._apply_constraints()

    def _reg(self, constr: dict) -> float:
        return constr["measures"][self.dname]["regularity"]["item"]

    def _cvg(self, constr: dict) -> float:
        return constr["measures"][self.dname]["coverage"]["item"]

    def _apply_constraints(self) -> None:

        for c_id, constr in constraints.items():

            # retrieve acceptable constraint linkers
            # (will become unacceptable for the L2P constraints)
            constr_linkers: list = constr["acceptable_linkers"].copy()
            # remove allomorphs; -(")er- is merged in the constraint file
            # NB! but not in the applicability indices
            for allo in ["_+n_", "_+ns_"]:
                if allo in constr_linkers:
                    constr_linkers.remove(allo)
                    constr_linkers.append(allo.replace("+n", "+en"))
            constr_linkers = list(set(constr_linkers))

            # P2L: "generative" constraints that contribute 
            # to the acceptable linkers
            if c_id.startswith("p2l:"):
                if self.appl_row.get(f"{c_id}_is_applicable", False):
                    for linker in constr_linkers:
                        self.acceptable_linkers[linker].append({
                            "c_id": c_id,
                            "item_reg": self._reg(constr),
                            "item_cvg": self._cvg(constr),
                        })

            # unacceptable linkers: restricted by the L2P constraints;
            # here, we stick to a similar logic that the original L2P checks 
            # implemented, but now, since we are not validating actual linkers
            # but restrict potential linkers, we disregard the former
            elif c_id.startswith("l2p:"):
                is_presat_func = getattr(
                    l2p_satisfaction_checkers,
                    f"{constr["func_alias"]}_is_presatisfied",
                    lambda _: False # False if not implemented
                )
                # this check is required for some L2P linkers
                # that impose not general but targeted restrictions;
                # for example, there are some restrictions on
                # simplex nouns that may attach -s-, which leaves
                # complex nouns undefined with regard to this linker;
                # in order to not mark complex nouns in this example
                # as false negative (as if they cannot attach -s-
                # since they are not simplex), we run this additional check
                if not is_presat_func(self.comp):
                    continue
                is_sat_func = getattr(
                    l2p_satisfaction_checkers,
                    f"{constr["func_alias"]}_is_satisfied"
                )
                if not is_sat_func(self.comp):
                    # the condition that would allow the corresponding
                    # linker are not satisfied => the linker may not
                    # appear with this N1; e.g. 'Hund' would violate
                    # the er-paradigmaticity condition, and so
                    # -(")er- would be prohibited to appear with this N1
                    for linker in constr_linkers:
                        self.unacceptable_linkers[linker].append({
                            "c_id": c_id,
                            "item_reg": self._reg(constr),
                            "item_cvg": self._cvg(constr),
                        })

    def predict(
        self,
        *,
        method: Literal["albright_hayes"],
        constr_cooperation: bool,
        restrict_with_l2p: bool
    ):
        if method == "albright_hayes":
            return self._albright_hayes(
                constr_cooperation=constr_cooperation,
                restrict_with_l2p=restrict_with_l2p
            )

    def _albright_hayes(
        self,
        constr_cooperation: bool,
        restrict_with_l2p: bool
    ):
        
        # first, sort acceptable linkers by item reg
        linker_scoreboard = {}
        for linker, c_ids_regs in self.acceptable_linkers.items():
            if not constr_cooperation:
                # original Albright & Bayes method:
                # take the max confidence
                most_conf_constr = max(
                    c_ids_regs,
                    key=lambda x: x["item_reg"]
                )
                linker_scoreboard[linker] = {
                    "raw_conf": most_conf_constr["item_reg"],
                    "cvg": most_conf_constr["item_cvg"]
                }
            else:
                # sum all the confidences
                item_reg_sum, item_cvg_sum = 0, 0
                for c_id_reg in c_ids_regs:
                    item_reg_sum += c_id_reg["item_reg"]
                    item_cvg_sum += c_id_reg["item_cvg"]
                linker_scoreboard[linker] = {
                    "raw_conf": item_reg_sum,
                    "cvg": item_cvg_sum
                }

        if restrict_with_l2p:
            # remove linkers that are not acceptable
            # in accordance with the L2P constraints
            self.l2p_restrictions = defaultdict(list)
            for linker, c_ids_regs in self.unacceptable_linkers.items():
                if linker in linker_scoreboard:
                    if not constr_cooperation:
                        # original Albright & Bayes method:
                        # take the max confidence
                        most_conf_constr = max(
                            c_ids_regs,
                            key=lambda x: x["item_reg"]
                        )
                        raw_conf = most_conf_constr["item_reg"]
                        cvg = most_conf_constr["item_cvg"]
                    else:
                        # sum all the confidences
                        raw_conf, cvg = 0, 0
                        for c_id_reg in c_ids_regs:
                            item_reg_sum += c_id_reg["item_reg"]
                            item_cvg_sum += c_id_reg["item_cvg"]
                    # remove the linker only if the restrictive
                    # L2P constraint(s) has (have) a higher confidence
                    # than the P2L one(s); indirectly following Albright & Hayes 2002,
                    # choose the constraints with the higher coverage in case
                    # regularity is the same
                    if (
                        raw_conf > linker_scoreboard[linker]["raw_conf"] or
                        (
                            raw_conf == linker_scoreboard[linker]["raw_conf"] and
                            cvg > linker_scoreboard[linker]["cvg"]
                        )
                    ):
                        self.l2p_restrictions[linker].append(c_ids_regs)
                        linker_scoreboard.pop(linker)

        # indirectly following Albright & Hayes 2002,
        # prioritize the constraints with the higher coverage in case
        # regularity is the same
        self.linker_scoreboard = pd.DataFrame(linker_scoreboard).T  \
            .sort_values(by=["raw_conf", "cvg"], ascending=[False, False])

        # return all max generations
        max_conf = x.linker_scoreboard["raw_conf"].max()
        max_linkers = self.linker_scoreboard[self.linker_scoreboard["raw_conf"] == max_conf]
        max_cvg = max_linkers["cvg"].max()
        preds = max_linkers[max_linkers["cvg"] == max_cvg]

        return preds

In [13]:
appl_inds["gecodb_v07_gn"].iloc[0].name

'land_+es_regierung'

In [8]:
x = CompXConstraints(appl_inds["gecodb_v07_gn"].iloc[0], "gecodb_v07_gn")

In [9]:
x.acceptable_linkers

defaultdict(list,
            {'_': [{'c_id': 'def-0', 'item_reg': 0.573, 'item_cvg': 1.0},
              {'c_id': 'p2l:decl_cl:pl:#er-0/er|def-0/er',
               'item_reg': 0.815,
               'item_cvg': 0.021}],
             '_+=er_': [{'c_id': 'p2l:decl_cl:pl:#er-0/er|def-0/er',
               'item_reg': 0.815,
               'item_cvg': 0.021}],
             '_+s_': [{'c_id': 'p2l:tend:sonority-!s',
               'item_reg': 0.793,
               'item_cvg': 0.435}]})

In [14]:
x.unacceptable_linkers

defaultdict(list,
            {'_+en_': [{'c_id': 'l2p:en|par',
               'item_reg': 0.985,
               'item_cvg': 0.152}],
             '_+e_': [{'c_id': 'l2p:e|par',
               'item_reg': 0.968,
               'item_cvg': 0.009}],
             '_+=e_': [{'c_id': 'l2p:e_uml',
               'item_reg': 1.0,
               'item_cvg': 0.003}],
             '_+=_': [{'c_id': 'l2p:0_uml',
               'item_reg': 1.0,
               'item_cvg': 0.001}]})

In [15]:
pred = x.predict(
    method="albright_hayes",
    constr_cooperation=False,
    restrict_with_l2p=True
)

pred

,raw_conf,cvg
_,0.815,0.021
_+=er_,0.815,0.021


In [16]:
pred = x.predict(
    method="albright_hayes",
    constr_cooperation=True,
    restrict_with_l2p=True
)

pred

,raw_conf,cvg
_,1.388,1.021


## Baseline: Random Distribution-Aware Sampling

In [ ]:
unique_linkers = gecodb["linker"].unique()
linkers = gecodb["linker"].tolist()
linker_distr = pd.Series({
    l: linkers.count(l) / N
    for l in linkers
})
linker_distr.sort_values(ascending=False)

_         0.573485
_+s_      0.268907
_+n_      0.099906
_+en_     0.016232
_+es_     0.015007
_+er_     0.008555
_+e_      0.006452
_+=er_    0.005738
_+ns_     0.003165
_+=e_     0.001940
_+ens_    0.000408
_+=_      0.000204
dtype: float64

In [ ]:
rand_linkers = rng.choice(linker_distr.index, size=N, replace=True, p=linker_distr)
calc_metrics(gecodb["linker"], rand_linkers)

(0.4127327670695851, 0.08421396647185787)